# 第2回 宿題（発展）：用語集Q&AのLoRA学習と検索方式の比較

講座の用語集から作成したQ&Aデータを使い、LoRA学習の前後で回答がどう変わるかを確認する。
さらに、用語名による単純検索と比較し、学習させる方法のメリット・デメリットを考察する。

## 実験の条件

- 学習データ：messages形式のQ&A 116件を、学習用104件・検証用12件に分割。
- モデルへの入力：質問のみ。用語集・正解例・SYSTEMプロンプトは渡さない。
- モデルの出力：通常の説明文。JSON形式や確認フラグは要求しない。
- 正解例と採点ポイント：goldに保存し、回答の評価に使用する。

## 評価の構成

- 本評価15問：用語を直接尋ねる質問、言い換えた質問、用語集にない用語の質問を各5問。
- 補足実験3問：複数の用語や、その関係を尋ねる質問。本評価とは分けて扱う。
- 比較対象：学習前のモデル、学習後のモデル、用語集の単純検索。

## 実行時の注意

最初から実行することで、モデルの学習と回答生成を行い、結果を保存する。
保存済みの結果を確認するだけの場合は、再学習は不要。
既存の結果を残して再実験する場合は、保存先を分ける。

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
%pip uninstall -y cudf-cu13
PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
    # "pyarrow": "23.0.0",  # cudfが要求する19以上・24未満にそろえる
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

Found existing installation: cudf-cu13 26.6.0
Uninstalling cudf-cu13-26.6.0:
  Successfully uninstalled cudf-cu13-26.6.0
{'transformers': '5.21.0', 'peft': '0.21.2', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

モデル、学習条件、回答生成条件をCONFIGにまとめる。

- 基盤モデル：Qwen/Qwen3.5-0.8B
- revision：使用するモデルの版を固定する。
- 学習用データ：104件
- エポック数：10
- 学習率：2e-4
- LoRAのランク：16
- LoRAのalpha：32
- 学習時の最大系列長：512トークン
- 回答の最大生成長：256トークン
- 回答生成：サンプリングを使わない貪欲生成とする。

学習前後で質問と生成条件をそろえ、回答の変化を比較する。
学習済みアダプタと回答結果はGoogle Driveに保存する。

なお、seedを指定しても、実行環境によって学習結果に差が生じる場合がある。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 10,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分） デフォルトは20, 過学習を抑えるために10へ変更
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,    # デフォルト：96
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = True  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

Mounted at /content/drive
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 10,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 256,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


### Q&Aデータの読み込み


In [ ]:
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

data_path = Path("/content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_sft.jsonl")
with data_path.open(encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("Q&A件数:", len(records))
for message in records[0]["messages"]:
    print(message["role"], "→", message["content"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Q&A件数: 116
user → 「AIエンジニアリング」とは何ですか？
assistant → 学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。


### Q&Aデータの質問（Q）の確認

In [ ]:
# 1用語につき2件あるので、1件おきに質問を表示する
for i, record in enumerate(records[::2], start=1):
    print(i, record["messages"][0]["content"])

1 「AIエンジニアリング」とは何ですか？
2 「評価と改善のサイクル」とは何ですか？
3 「LLM（大規模言語モデル）」とは何ですか？
4 「トークン」とは何ですか？
5 「コンテキストウィンドウ」とは何ですか？
6 「プロンプト」とは何ですか？
7 「システムプロンプト」とは何ですか？
8 「temperature」とは何ですか？
9 「ハルシネーション」とは何ですか？
10 「API」とは何ですか？
11 「APIキー」とは何ですか？
12 「環境変数」とは何ですか？
13 「レート制限」とは何ですか？
14 「GitHub Codespaces」とは何ですか？
15 「devcontainer」とは何ですか？
16 「Codespaces secret」とは何ですか？
17 「fork」とは何ですか？
18 「ブランチ」とは何ですか？
19 「プルリクエスト（PR）」とは何ですか？
20 「GitHub Copilot」とは何ですか？
21 「コード補完」とは何ですか？
22 「コーディングエージェント」とは何ですか？
23 「事前学習」とは何ですか？
24 「ファインチューニング」とは何ですか？
25 「教師ありファインチューニング（SFT）」とは何ですか？
26 「PEFT」とは何ですか？
27 「LoRA」とは何ですか？
28 「ランク（LoRAのr）」とは何ですか？
29 「QLoRA」とは何ですか？
30 「量子化」とは何ですか？
31 「学習率」とは何ですか？
32 「エポック」とは何ですか？
33 「過学習」とは何ですか？
34 「Hugging Face Hub」とは何ですか？
35 「AIエージェント」とは何ですか？
36 「エージェントループ」とは何ですか？
37 「ツール（Tool use / Function calling）」とは何ですか？
38 「ツールの説明文」とは何ですか？
39 「構造化出力」とは何ですか？
40 「ハーネス」とは何ですか？
41 「Pi」とは何ですか？
42 「Pydantic AI」とは何ですか？
43 「ADK（Agent Development Kit）」とは何ですか？
44 「MCP（Model Context Protocol）」とは何ですか？
45 「ガードレール」とは何ですか？
46 「Hum

## 2. 評価質問と正解の作成

本評価として、次の質問を各5問、合計15問用意する。

- 用語を直接尋ねる質問
- 同じ用語を言い換えて尋ねる質問
- 用語集にない用語を尋ねる質問

各質問の正解例と採点ポイントをgoldに保存する。
goldは評価に使い、モデルには渡さない。
用語集にない質問も、回答内容が正しければ正解とする。

複数の用語や関係を尋ねる3問も、補足実験として用意する。
本評価15問をmain_test、補足3問をadditional_testとして分けて扱う。
testには両方を合わせた18問を入れる。

直接質問は学習データに近い表現である。
補足3問は本評価の結果を見てから追加したため、別にまとめる。

### 評価データの作成

In [ ]:
import pandas as pd
# 用語と説明文の対応表
definitions = {
    record["messages"][0]["content"]
        .removeprefix("「").removesuffix("」とは何ですか？"):
    record["messages"][1]["content"]
    for record in records
    if record["messages"][0]["content"].startswith("「")
}

# 用語集にある質問・言い換え
EVAL_QUESTIONS = [
    # 用語集にある質問
    ("用語集にある", "トークン", "トークンとは何ですか？"),
    ("用語集にある", "ハルシネーション", "ハルシネーションとは何ですか？"),
    ("用語集にある", "LoRA", "LoRAとは何ですか？"),
    ("用語集にある", "学習率", "学習率とは何ですか？"),
    ("用語集にある", "過学習", "過学習とは何ですか？"),

    # 同じ用語を、学習時と異なる聞き方で質問
    ("言い換え", "トークン", "LLMが文章を扱う際の「トークン」は、どのような単位ですか？"),
    ("言い換え", "ハルシネーション", "AIが事実と異なる内容をもっともらしく答える現象を説明してください。"),
    ("言い換え", "LoRA", "LoRAでは、モデル全体を更新する代わりに何を学習するのですか？"),
    ("言い換え", "学習率", "学習率は、モデルの学習中に何を調整する値ですか？"),
    ("言い換え", "過学習", "学習データにはよく答えられるのに、新しいデータで性能が落ちる状態を説明してください。"),
]

test = []
for category, term, question in EVAL_QUESTIONS:
    test.append({
        "category": category,
        "question": question,
        "answer": definitions[term],
    })

# 用語集にない質問
# 正解の説明文が用語集にないため、回答内容は別途確認する
UNSEEN_QUESTIONS = [
    "RAGとは何ですか？",
    "埋め込みベクトルとは何ですか？",
    "ベクトルデータベースとは何ですか？",
    "検索の適合率とは何ですか？",
    "検索の再現率とは何ですか？",
]

for question in UNSEEN_QUESTIONS:
    test.append({
        "category": "用語集にない",
        "question": question,
        "answer": None,
    })

# 評価用の正解例
# 用語集にない5問についても、採点用の正解を用意する
EXTRA_GOLD = {
    "RAGとは何ですか？":
        "質問に関連する情報を外部の文書などから検索し、"
        "検索結果を根拠として言語モデルが回答を生成する方法。",

    "埋め込みベクトルとは何ですか？":
        "テキストや画像などの特徴を数値のベクトルで表現したもの。"
        "ベクトル間の類似度を使い、意味や特徴が近いデータを検索できる。",

    "ベクトルデータベースとは何ですか？":
        "ベクトルを保存・管理し、入力したベクトルに類似する"
        "ベクトルを検索するためのデータベース。",

    "検索の適合率とは何ですか？":
        "検索で取得した項目のうち、質問に関連する項目の割合。"
        "関連する取得項目数を、取得した全項目数で割って求める。",

    "検索の再現率とは何ですか？":
        "関連する項目全体のうち、検索で取得できた項目の割合。"
        "関連する取得項目数を、関連する全項目数で割って求める。",
}

# 各質問で確認する要点
# 単語の一致ではなく、この意味を説明できているかを人が確認する
KEY_POINTS = [
    ["LLMがテキストを扱う単位", "単語・文字・その断片などに分割される"],
    ["事実と異なる情報や存在しない情報を生成する", "もっともらしい回答になる"],
    ["基盤モデルの重みを固定する", "更新分を表す低ランクの行列を学習する"],
    ["モデルのパラメータ更新の大きさを調整する"],
    ["学習データへの過度な適合", "未知のデータへの性能が低下する"],

    ["LLMがテキストを扱う単位", "文字数と常に一致するわけではない"],
    ["事実と異なる情報をもっともらしく生成する現象"],
    ["基盤モデルの重みを固定する", "追加した低ランクの行列を学習する"],
    ["モデルのパラメータ更新の大きさを調整する"],
    ["過学習である", "学習データへの過度な適合により未知のデータで性能が落ちる"],

    ["関連情報を検索する", "検索結果を使って言語モデルが回答を生成する"],
    ["データの特徴を数値ベクトルで表す", "類似度の計算や検索に利用できる"],
    ["ベクトルを保存・管理する", "類似するベクトルを検索する"],
    ["分母は検索で取得した全項目", "分子はそのうち関連する項目"],
    ["分母は関連する全項目", "分子はそのうち検索で取得できた項目"],
]

assert len(test) == len(KEY_POINTS) == 15

gold = {}

for ex, points in zip(test, KEY_POINTS):
    question = ex["question"]
    reference = EXTRA_GOLD.get(question, ex["answer"])
    assert reference is not None, f"正解が未設定です: {question}"

    gold[question] = {
        "reference_answer": reference,
        "key_points": points,
        "in_glossary": ex["category"] != "用語集にない",
    }

# 保存
gold_path = data_path.parent / "glossary_gold.json"
gold_path.write_text(
    json.dumps(gold, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("正解データ:", len(gold), "問")
print("保存先:", gold_path)

display(pd.DataFrame(test))

正解データ: 15 問
保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_gold.json


,category,question,answer
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...
6,言い換え,AIが事実と異なる内容をもっともらしく答える現象を説明してください。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...
7,言い換え,LoRAでは、モデル全体を更新する代わりに何を学習するのですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...
8,言い換え,学習率は、モデルの学習中に何を調整する値ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...
9,言い換え,学習データにはよく答えられるのに、新しいデータで性能が落ちる状態を説明してください。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...


In [ ]:
# @title A：追加3問の質問・正解を定義する
# 複数の用語を扱う追加評価
# 学習には使わず、学習前・学習後・検索版の比較に使う
ADDITIONAL_EVAL = [
    {
        "category": "用語集にある",
        "question": "学習率とエポックは、それぞれ学習の何を調整しますか？",
        "answer": (
            "学習率は1回の更新でモデルのパラメータをどれだけ動かすかを決める。"
            "エポック数は学習データ全体を何周するかを決める。"
        ),
        "key_points": [
            "学習率はパラメータ更新の大きさを調整する",
            "エポック数は学習データ全体を繰り返し学習する回数を調整する",
        ],
        "in_glossary": True,
    },
    {
        "category": "言い換え",
        "question": (
            "モデルの重みを一度にどれだけ変えるかという設定と、"
            "学習データを何周させるかという設定は、どう違いますか？"
        ),
        "answer": (
            "前者は学習率、後者はエポック数。"
            "学習率は1回のパラメータ更新の大きさを決め、"
            "エポック数は学習データ全体を繰り返し学習する回数を決める。"
        ),
        "key_points": [
            "前者を学習率、後者をエポック数と説明する",
            "更新の大きさと学習を繰り返す回数の違いを説明する",
        ],
        "in_glossary": True,
    },
    {
        "category": "用語集にない",
        "question": "検索の適合率と再現率は、それぞれ何を測る指標ですか？",
        "answer": (
            "適合率は検索で取得した項目のうち、関連する項目の割合。"
            "再現率は関連する項目全体のうち、検索で取得できた項目の割合。"
        ),
        "key_points": [
            "適合率の分母は取得した全項目、分子は関連する取得項目",
            "再現率の分母は関連する全項目、分子は関連する取得項目",
        ],
        "in_glossary": False,
    },
]

# 再実行しても追加質問が重複しないようにする
additional_questions = {ex["question"] for ex in ADDITIONAL_EVAL}
test = [ex for ex in test if ex["question"] not in additional_questions]

for ex in test:
    ex["evaluation_group"] = "既存15問"

for ex in ADDITIONAL_EVAL:
    test.append({
        "category": ex["category"],
        "question": ex["question"],
        "answer": ex["answer"],
        "evaluation_group": "追加3問",
    })

    gold[ex["question"]] = {
        "reference_answer": ex["answer"],
        "key_points": ex["key_points"],
        "in_glossary": ex["in_glossary"],
    }

# 件数・重複・正解の対応を確認
assert len(test) == 18, f"評価用質問が18問ではありません: {len(test)}"
assert len({ex["question"] for ex in test}) == len(test)
assert all(ex["question"] in gold for ex in test)

# 今回の評価対象に正解データをそろえる
gold = {ex["question"]: gold[ex["question"]] for ex in test}

# 評価データと正解を保存
test_path = data_path.parent / "glossary_test.json"
gold_path = data_path.parent / "glossary_gold.json"

test_path.write_text(
    json.dumps(test, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
gold_path.write_text(
    json.dumps(gold, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame(test))
print("評価用質問:", len(test))
print("正解データ:", len(gold))
print("保存先:", test_path)
print("保存先:", gold_path)

# 本評価と補足実験を分ける
main_test = [
    ex for ex in test
    if ex["evaluation_group"] == "既存15問"
]
additional_test = [
    ex for ex in test
    if ex["evaluation_group"] == "追加3問"
]

assert len(main_test) == 15
assert len(additional_test) == 3

print("本評価:", len(main_test), "問")
print("補足実験:", len(additional_test), "問")

,category,question,answer,evaluation_group
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,既存15問
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,既存15問
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,既存15問
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,既存15問
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,既存15問
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,既存15問
6,言い換え,AIが事実と異なる内容をもっともらしく答える現象を説明してください。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,既存15問
7,言い換え,LoRAでは、モデル全体を更新する代わりに何を学習するのですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,既存15問
8,言い換え,学習率は、モデルの学習中に何を調整する値ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,既存15問
9,言い換え,学習データにはよく答えられるのに、新しいデータで性能が落ちる状態を説明してください。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,既存15問


評価用質問: 18
正解データ: 18
保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_test.json
保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_gold.json
本評価: 15 問
補足実験: 3 問


## 3. モデルの読み込みと学習データの準備

CONFIGで指定した基盤モデルを、量子化せずに読み込む。

用語集から作成した116件のQ&Aを、
学習用104件と検証用12件に分ける。
データはmessages形式のまま学習に使う。

学習用と検証用には、同じ用語を異なる聞き方で尋ねる例が含まれる。

回答生成時は質問だけを渡す。
SYSTEMプロンプト、用語集本文、正解例は渡さず、
JSON形式や確認フラグも要求しない。

In [ ]:
# @title モデルの読み込み
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

tok = AutoTokenizer.from_pretrained(
    CONFIG["base_model"],
    revision=CONFIG["revision"],
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# 基盤モデルを読み込む（量子化なし・32bit）
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    revision=CONFIG["revision"],
    dtype=torch.float32,
).to(DEVICE)

model.config.use_cache = True
model.eval()

BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print(
    "loaded:", CONFIG["base_model"],
    "| device:", DEVICE,
    "| 重みのメモリ(GB):", BASE_MEMORY_GB,
)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [ ]:
# @title 学習・検証・評価用データの作成（学習用：104件、検証用：12件、評価用：15件）
import random
from datasets import Dataset

# 1用語につき2種類の質問が、順番に並んでいるデータを使用
# 1種類目は全用語を学習用にする
train_records = list(records[::2])

# 2種類目の質問を、学習用と検証用に分ける
remaining_records = list(records[1::2])

rng = random.Random(CONFIG["seed"])
rng.shuffle(remaining_records)

valid_records = remaining_records[:12]
train_records.extend(remaining_records[12:])
rng.shuffle(train_records)

# すでに messages 形式なので、FAQ用の to_chat() は不要
train_ds = Dataset.from_list(train_records)
valid_ds = Dataset.from_list(valid_records)

# 件数を確認
assert len(train_ds) + len(valid_ds) == len(records)

print("学習用:", len(train_ds), "件")
print("検証用:", len(valid_ds), "件")
print("評価用:", len(test), "問")

# 学習データの例を確認
print("\n学習データの例")
for message in train_ds[0]["messages"]:
    print(message["role"], "→", message["content"])

学習用: 104 件
検証用: 12 件
評価用: 18 問

学習データの例
user → 「ガードレール」とは何ですか？
assistant → エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。


## 4. 学習前の回答取得とLoRA学習

まず、学習前のモデルに本評価15問を質問し、
回答をglossary_before.jsonに保存する。
回答生成には、学習前後で共通のgenerate_glossaryを使う。

次に、学習用104件のQ&AでLoRA学習を行う。
基盤モデルの重みは固定し、追加したアダプタだけを学習する。
学習条件はCONFIGで指定する。

学習中のloss、学習時間、更新回数を確認する。
lossが下がっても、回答が正しくなるとは限らないため、
学習後の回答を別途評価する。

学習済みアダプタはGoogle Driveに保存する。
学習後のモデルはtunedで参照する。
学習前の回答を取り直す場合は、
tuned.disable_adapter()でアダプタを一時的に無効にする。

In [ ]:
# @title 追加したgenerate_glossary処理
import json
import torch
import pandas as pd

# 学習前後で共通の生成処理
def generate_glossary(model, question):
    messages = [
        {"role": "user", "content": question},
    ]

    text = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tok(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    return tok.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()


# 学習前の回答を取得
model.eval()
before_rows = []

for ex in main_test:
    before_raw = generate_glossary(model, ex["question"])
    before_rows.append({
        "category": ex["category"],
        "question": ex["question"],
        "answer": ex["answer"],
        "before": before_raw,
    })

display(pd.DataFrame(before_rows))


# 読み込んだJSONLと同じDriveフォルダに保存
before_path = data_path.parent / "glossary_before.json"

before_path.write_text(
    json.dumps(before_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("保存先:", before_path)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


,category,question,answer,before
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,**トークン（Token）** は、インターネット上で通信や処理を行う際に使用される**単一...
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,**ハルシネーション**（Hulshinēshon）は、物理学、特に**量子力学**において...
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つ...
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,"**学習率（Learning Rate, LR）**とは、機械学習において、モデルがデータか..."
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,**過学習（Overfitting）**とは、機械学習や統計学習において、モデルが学習したデ...
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,LLM（大規模言語モデル）が文章を扱う際、「トークン」は、**文字列（テキスト）の単位**で...
6,言い換え,AIが事実と異なる内容をもっともらしく答える現象を説明してください。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,AI が事実と異なる内容をもっともらしく答える現象は、**「誤解された事実（Falsehoo...
7,言い換え,LoRAでは、モデル全体を更新する代わりに何を学習するのですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,LoRA（Low-Rank Adaptation）では、モデル全体（全層）を更新するのではな...
8,言い換え,学習率は、モデルの学習中に何を調整する値ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,学習率は、モデルの学習中に**学習率（learning rate）**を調整する値です。\n...
9,言い換え,学習データにはよく答えられるのに、新しいデータで性能が落ちる状態を説明してください。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,学習データに「よく答えられる」のに、新しいデータで性能が落ちる状態を**「過学習（Overf...


保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_before.json


In [ ]:
# @title データの確認（質問、正解例、学習前）

USE_DRIVE = True

# 今回の発展課題専用の保存先
CONFIG["drive_dir"] = str(data_path.parent / "glossary_lora")
Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)

# 学習前の最終確認
print("学習用:", len(train_ds))
print("検証用:", len(valid_ds))
print("評価用:", len(test))
print("保存済みの学習前回答:", len(before_rows))

for i, row in enumerate(before_rows):
    print(f"\n【{i}】{row['question']}")
    print("正解例:", row["answer"])
    print("学習前:", row["before"])
    print("-" * 60)

学習用: 104
検証用: 12
評価用: 18
保存済みの学習前回答: 15

【0】トークンとは何ですか？
正解例: LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
学習前: **トークン（Token）** は、インターネット上で通信や処理を行う際に使用される**単一のデータ単位**です。

簡単に言うと、「通信の単位」と「処理の単位」を指します。例えば、Web サイトのユーザー登録時に必要になる「ユーザー名」や、決済システムで使われる「クレジットカード番号」などが、トークンの例です。

トークンがどのように機能するかは、そのシステムによって異なりますが、一般的には以下の 3 つの役割を果たします。

### 1. 認証（認証トークン）
最も一般的な用途は、ユーザーが特定のサービスにログインできるかどうかを確認するものです。
*   **例**: Google の Google アカウント、Facebook の Facebook アカウント、Apple の Apple ID。
*   **仕組み**: 認証トークンは、ユーザーが特定のサービスにログインする際に、そのサービスにアクセスできるかどうかを判断するために使用されます。

### 2. 取引処理（取引トークン）
決済や取引を行う際、取引の完了や確認のために使用されます。
*   **例**: 銀行の ATM での支払い、クレジットカード決済、仮想通貨取引。
*   **仕組み**: 取引が完了した際に発行されるトークン（または取引履歴）は
------------------------------------------------------------

【1】ハルシネーションとは何ですか？
正解例: LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
学習前: **ハルシネーション**（Hulshinēshon）は、物理学、特に**量子力学**にお

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# def to_chat(example):
#     # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
#     return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

# train_ds = Dataset.from_list([to_chat(e) for e in train])
# valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.232964
20,2.207715
30,1.492058
40,0.977735
50,0.552444
60,0.294726
70,0.165199
80,0.146798
90,0.117466
100,0.101157


学習時間: 420 秒 | ステップ数: 130 | エポック: 10.0


### 4b. 接続が切れた場合

Colabの接続が切れて実行環境がリセットされると、
読み込んだモデルや変数が消える場合がある。
Google Driveに保存した結果は残る。

- 保存済みの回答を比較するだけなら、再学習は不要である。
- 新しい質問に回答させる場合は、モデルと保存済みの学習結果を読み込む。
- 設定を変えて学習し直す場合は、以前の結果を上書きしないよう保存先を分ける。

## 5. 学習後の回答と比較

学習後のモデルに、学習前と同じ15問を質問する。
質問と回答生成の設定をそろえ、学習によって回答がどう変わったかを確認する。

学習後の回答はglossary_after.jsonに保存する。

比較では、次の点を確認する。

- 用語の説明が正しくなったか。
- 言い換えた質問にも回答できるか。
- 用語集にない質問に、誤った説明をしていないか。
- 学習前には正しかった回答が、学習後に誤っていないか。

追加3問は補足実験として扱い、本評価15問とは分けて確認する。

In [ ]:
import json
import time
import pandas as pd

# 学習後のモデルを回答生成用の状態にする
tuned = trainer.model
tuned.config.use_cache = True
tuned.eval()

# 学習前と同じ質問・同じ生成処理で回答を取得
after_rows = []
t0 = time.time()

for b in before_rows:
    after_raw = generate_glossary(tuned, b["question"])
    after_rows.append({
        "category": b["category"],
        "question": b["question"],
        "answer": b["answer"],
        "after": after_raw,
    })

print(f"評価件数: {len(after_rows)} 問")
print(f"回答生成時間: {time.time() - t0:.1f} 秒")

# 学習前後の比較表
detail = pd.DataFrame([
    {
        "区分": b["category"],
        "質問": b["question"],
        "正解例": b["answer"],
        "学習前の出力": b["before"],
        "学習後の出力": a["after"],
    }
    for b, a in zip(before_rows, after_rows)
])

pd.set_option("display.max_colwidth", None)
display(detail)

# Driveに保存
after_path = data_path.parent / "glossary_after.json"
after_path.write_text(
    json.dumps(after_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

compare_path = data_path.parent / "glossary_compare.json"
compare_path.write_text(
    json.dumps(
        detail.to_dict(orient="records"),
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("学習後回答の保存先:", after_path)
print("比較表の保存先:", compare_path)

評価件数: 15 問
回答生成時間: 64.1 秒


,区分,質問,正解例,学習前の出力,学習後の出力
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,**トークン（Token）** は、インターネット上で通信や処理を行う際に使用される**単一のデータ単位**です。\n\n簡単に言うと、「通信の単位」と「処理の単位」を指します。例えば、Web サイトのユーザー登録時に必要になる「ユーザー名」や、決済システムで使われる「クレジットカード番号」などが、トークンの例です。\n\nトークンがどのように機能するかは、そのシステムによって異なりますが、一般的には以下の 3 つの役割を果たします。\n\n### 1. 認証（認証トークン）\n最も一般的な用途は、ユーザーが特定のサービスにログインできるかどうかを確認するものです。\n* **例**: Google の Google アカウント、Facebook の Facebook アカウント、Apple の Apple ID。\n* **仕組み**: 認証トークンは、ユーザーが特定のサービスにログインする際に、そのサービスにアクセスできるかどうかを判断するために使用されます。\n\n### 2. 取引処理（取引トークン）\n決済や取引を行う際、取引の完了や確認のために使用されます。\n* **例**: 銀行の ATM での支払い、クレジットカード決済、仮想通貨取引。\n* **仕組み**: 取引が完了した際に発行されるトークン（または取引履歴）は,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,**ハルシネーション**（Hulshinēshon）は、物理学、特に**量子力学**において、**「量子の波動性」**という性質を表現する概念です。\n\nこの言葉は、1927 年にドイツの物理学者**エドゥアルド・シュミッン（Eduard Schumacher）**によって初めて正式に定義されました。その意味は以下の通りです。\n\n### 1. 基本定義\nシュミッンが定義したハルシネーションは、**「粒子が空間を通過する際、その位置や方向が不確定であること」**を指します。\n\n具体的には、以下の 2 つの性質を同時に満たす状態を指します。\n* **確率分布の広がり**: 粒子がどの位置にいるか、どの方向に移動する確率分布が広がりすぎている状態。\n* **波の性質**: 粒子が波として振動している状態。\n\n### 2. 物理的な意味\nこの現象は、**「粒子としての確実な位置と、波としての広がり」**の両方を同時に持っている状態を指します。\n\n* **粒子としての性質**:,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「モデルの学習を細かく制御して、特定の知識や文脈に特化した小さなモデルを生成する技術」**です。\n\n### 主な特徴と仕組み\n\nLoRA は、従来の大規模モデル（例えば GPT-4）を「大規模な基盤」として学習させるのではなく、その基盤を「小さな基盤」として学習させるアプローチです。\n\n1. **学習の規模**:\n * **大規模モデル**: 数百万から数千万のパラメータを学習します。\n * **LoRA**: 数千万から数億のパラメータを学習します。\n * 学習コストが大幅に削減され、トレーニングが非常に効率的に行えます。\n\n2. **学習の手法**:\n LoRA は、**「パラメータの結合（結合学習）」**という手法を使います。\n * 大規模モデルの学習時に、すべてのパラメータが同時に更新されます。\n * LoRA では、特定の知識（例：特定の専門用語、特定の文脈、特定のタ,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,"**学習率（Learning Rate, LR）**とは、機械学習において、モデルがデータから学習した結果、**誤り（エラー）を最小化するために、入力値を修正するステップの大きさを指します。**\n\nより具体的に言うと、学習率とは「モデルが学習した結果、入力値を修正する際のステップサイズ（ステップの大きさ）」のことです。\n\n### 学習率の役割と仕組み\n\n学習率の役割は主に以下の 2 つの点にあります。\n\n1. **学習の速度を制御する**\n 学習率を大きくすると、モデルが学習する速度が速くなりますが、学習が過剰に進むリスクが高まります。逆に小さくすると学習がゆっくり進みますが、学習が停滞するリスクも高まります。学習率は、学習の速度を「バランス」に調整する重要なパラメータです。\n\n2. **誤り（エラー）を最小化する**\n 学習率は、モデルが学習した結果、入力値を修正する際に、その修正の大きさを決定します。修正が小さければ、モデルが学習した結果をより正確に反映できず、学習が停滞します。修正が大きければ、学習が過剰",1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,**過学習（Overfitting）**とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに適用できない**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真のデータでもうまくいかない」状態です。\n\n### 過学習の主な特徴\n1. **過剰適合**: 学習したモデルは、訓練データ（学習したデータ）の分布に非常に近い形状を維持し、真のデータ（未学習したデータ）の分布と異なる特徴を学習してしまうことがあります。\n2. **精度の低下**: 訓練データだけで得られた精度が、真のデータに対しては大幅に低下します。\n3. **欠陥の増加**: 学習したモデルは、訓練データに含まれる「過剰なパターン」や「ノイズ」に敏感になり、真のデータに適用すると誤りが増加します。\n\n### 過学習の原因\n過学習は、以下の要因が関与します。\n* **欠き欠けのデータ**: 学習対象となるデータが非常に少ない場合（欠き欠けデータ）。\n* **過学習のデータ**: 学習対象,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,LLM（大規模言語モデル）が文章を扱う際、「トークン」は、**文字列（テキスト）の単位**です。\n\n具体的には、テキストを「文字列」として扱うため、**文字の個数**という単位で定義されます。例えば、日本語の「こんにちは」は 3 つの文字（「こ」、「に」、「ほ」）の 3 つのトークンとして扱われます。\n\nこの「トークン」の単位は、以下の 3 つの要素から構成されます：\n\n1. **文字（Character）**：テキストに含まれる基本単位。\n2. **言語（Language）**：テキストに含まれる言語の種別。\n3. **順序（Order）**：テキストに含まれる順序関係。\n\nつまり、LLM はテキストを「文字列」として処理し、その

学習後回答の保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_after.json
比較表の保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_compare.json


In [ ]:
# @title B：追加3問の回答を生成する
# 保存済みアダプタの場所を確認する（GPU不要）
adapter_configs = list(
    data_path.parent.rglob("adapter_config.json")
)

for path in adapter_configs:
    print("アダプタの保存先:", path.parent)

if not adapter_configs:
    print("このフォルダ内には保存済みアダプタが見つかりませんでした")

# 追加3問の回答を生成する
assert "tuned" in globals(), (
    "学習済みモデルが読み込まれていません。"
    "保存済みアダプタを読み込んでから実行してください。"
)

tuned.eval()
additional_rows = []

for ex in additional_test:
    question = ex["question"]

    # アダプタを無効にして、基盤モデルの回答を取得する
    with tuned.disable_adapter():
        before_raw = generate_glossary(tuned, question)

    # アダプタを有効にして、学習後の回答を取得する
    after_raw = generate_glossary(tuned, question)

    additional_rows.append({
        "category": ex["category"],
        "question": question,
        "reference_answer": gold[question]["reference_answer"],
        "before": before_raw,
        "after": after_raw,
    })

# 回答生成が終わってから、3問そろっているか確認する
assert len(additional_rows) == 3, (
    "追加3問の回答がそろっていないため保存しません"
)

# 既存15問の保存ファイルとは分けて保存する
additional_path = data_path.parent / "glossary_additional_compare.json"
additional_path.write_text(
    json.dumps(additional_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

for i, row in enumerate(additional_rows, start=15):
    print(f"\n【{i}】{row['question']}")
    print("正解例:", row["reference_answer"])
    print("\n学習前:", row["before"])
    print("\n学習後:", row["after"])
    print("-" * 60)

print("保存先:", additional_path)

アダプタの保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_lora/adapter

【15】学習率とエポックは、それぞれ学習の何を調整しますか？
正解例: 学習率は1回の更新でモデルのパラメータをどれだけ動かすかを決める。エポック数は学習データ全体を何周するかを決める。

学習前: 学習率（Learning Rate）とエポック（Epoch）は、学習の**「速度」**と**「回数」**を調整する二つの重要なパラメータです。それぞれが学習のどの段階で役割を果たすか、明確に区別する必要があります。

### 学習率（Learning Rate）
学習率は、**「学習の速度」**を調整します。

*   **役割**: 学習のステップ数（エポック）を決定するのではなく、**一度学習率を上げれば一度学習率を下げれば**、学習の進捗が速く遅く変わります。
*   **調整対象**: 学習の**速度**（学習の進捗）。
*   **影響**:
    *   学習率が**小さく**なら、学習は**ゆっくり**に進みます。
    *   学習率が**大きく**なら、学習は**速く**進みます。
*   **注意点**: 学習率が**過大**（学習が止まったり、過小（学習が進まなかったり））になると、学習の収束（安定化）が難しくなり、最適化が失敗する可能性があります。

### エポック（Epoch

学習後: 1回のエポックで動く大きさを決めるハイパーパラメータ。低いほど安定し、高いほど速度が上がる。同じ入力でも出力が変わる理由の一つ。
------------------------------------------------------------

【16】モデルの重みを一度にどれだけ変えるかという設定と、学習データを何周させるかという設定は、どう違いますか？
正解例: 前者は学習率、後者はエポック数。学習率は1回のパラメータ更新の大きさを決め、エポック数は学習データ全体を繰り返し学習する回数を決める。

学習前: モデルの重みを一度にどれだけ変えるか（**学習率**）と、学習データを何周させるか（**学習回数**）は、どちらも「学習の進捗」を表現する重要なパラメー

In [ ]:
# @title 検証用（GPU不使用時）
import json
import pandas as pd

# 保存済みの回答を読み込む
before_path = data_path.parent / "glossary_before.json"
after_path = data_path.parent / "glossary_after.json"

before_rows = json.loads(before_path.read_text(encoding="utf-8"))
after_rows = json.loads(after_path.read_text(encoding="utf-8"))

# 質問をキーにして、学習前後を対応させる
before_by_q = {row["question"]: row for row in before_rows}
after_by_q = {row["question"]: row for row in after_rows}

evaluation_rows = []

for ex in main_test:
    q = ex["question"]
    assert q in before_by_q, f"学習前の回答がありません: {q}"
    assert q in after_by_q, f"学習後の回答がありません: {q}"
    assert q in gold, f"正解データがありません: {q}"

    evaluation_rows.append({
        "区分": ex["category"],
        "質問": q,
        "正解例": gold[q]["reference_answer"],
        "採点ポイント": "／".join(gold[q]["key_points"]),
        "学習前": before_by_q[q]["before"],
        "学習後": after_by_q[q]["after"],
    })

evaluation_df = pd.DataFrame(evaluation_rows)

pd.set_option("display.max_colwidth", None)
display(evaluation_df)

print("比較できる質問数:", len(evaluation_df))

,区分,質問,正解例,採点ポイント,学習前,学習後
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,LLMがテキストを扱う単位／単語・文字・その断片などに分割される,**トークン（Token）** は、インターネット上で通信や処理を行う際に使用される**単一のデータ単位**です。\n\n簡単に言うと、「通信の単位」と「処理の単位」を指します。例えば、Web サイトのユーザー登録時に必要になる「ユーザー名」や、決済システムで使われる「クレジットカード番号」などが、トークンの例です。\n\nトークンがどのように機能するかは、そのシステムによって異なりますが、一般的には以下の 3 つの役割を果たします。\n\n### 1. 認証（認証トークン）\n最も一般的な用途は、ユーザーが特定のサービスにログインできるかどうかを確認するものです。\n* **例**: Google の Google アカウント、Facebook の Facebook アカウント、Apple の Apple ID。\n* **仕組み**: 認証トークンは、ユーザーが特定のサービスにログインする際に、そのサービスにアクセスできるかどうかを判断するために使用されます。\n\n### 2. 取引処理（取引トークン）\n決済や取引を行う際、取引の完了や確認のために使用されます。\n* **例**: 銀行の ATM での支払い、クレジットカード決済、仮想通貨取引。\n* **仕組み**: 取引が完了した際に発行されるトークン（または取引履歴）は,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,事実と異なる情報や存在しない情報を生成する／もっともらしい回答になる,**ハルシネーション**（Hulshinēshon）は、物理学、特に**量子力学**において、**「量子の波動性」**という性質を表現する概念です。\n\nこの言葉は、1927 年にドイツの物理学者**エドゥアルド・シュミッン（Eduard Schumacher）**によって初めて正式に定義されました。その意味は以下の通りです。\n\n### 1. 基本定義\nシュミッンが定義したハルシネーションは、**「粒子が空間を通過する際、その位置や方向が不確定であること」**を指します。\n\n具体的には、以下の 2 つの性質を同時に満たす状態を指します。\n* **確率分布の広がり**: 粒子がどの位置にいるか、どの方向に移動する確率分布が広がりすぎている状態。\n* **波の性質**: 粒子が波として振動している状態。\n\n### 2. 物理的な意味\nこの現象は、**「粒子としての確実な位置と、波としての広がり」**の両方を同時に持っている状態を指します。\n\n* **粒子としての性質**:,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,基盤モデルの重みを固定する／更新分を表す低ランクの行列を学習する,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つです。\n\n簡単に言うと、**「モデルの学習を細かく制御して、特定の知識や文脈に特化した小さなモデルを生成する技術」**です。\n\n### 主な特徴と仕組み\n\nLoRA は、従来の大規模モデル（例えば GPT-4）を「大規模な基盤」として学習させるのではなく、その基盤を「小さな基盤」として学習させるアプローチです。\n\n1. **学習の規模**:\n * **大規模モデル**: 数百万から数千万のパラメータを学習します。\n * **LoRA**: 数千万から数億のパラメータを学習します。\n * 学習コストが大幅に削減され、トレーニングが非常に効率的に行えます。\n\n2. **学習の手法**:\n LoRA は、**「パラメータの結合（結合学習）」**という手法を使います。\n * 大規模モデルの学習時に、すべてのパラメータが同時に更新されます。\n * LoRA では、特定の知識（例：特定の専門用語、特定の文脈、特定のタ,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,モデルのパラメータ更新の大きさを調整する,"**学習率（Learning Rate, LR）**とは、機械学習において、モデルがデータから学習した結果、**誤り（エラー）を最小化するために、入力値を修正するステップの大きさを指します。**\n\nより具体的に言うと、学習率とは「モデルが学習した結果、入力値を修正する際のステップサイズ（ステップの大きさ）」のことです。\n\n### 学習率の役割と仕組み\n\n学習率の役割は主に以下の 2 つの点にあります。\n\n1. **学習の速度を制御する**\n 学習率を大きくすると、モデルが学習する速度が速くなりますが、学習が過剰に進むリスクが高まります。逆に小さくすると学習がゆっくり進みますが、学習が停滞するリスクも高まります。学習率は、学習の速度を「バランス」に調整する重要なパラメータです。\n\n2. **誤り（エラー）を最小化する**\n 学習率は、モデルが学習した結果、入力値を修正する際に、その修正の大きさを決定します。修正が小さければ、モデルが学習した結果をより正確に反映できず、学習が停滞します。修正が大きければ、学習が過剰",1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,学習データへの過度な適合／未知のデータへの性能が低下する,**過学習（Overfitting）**とは、機械学習や統計学習において、モデルが学習したデータに**過剰に適合**し、**真のデータに適用できない**現象を指します。\n\n簡単に言うと、「学習したことが全部で、真のデータでもうまくいかない」状態です。\n\n### 過学習の主な特徴\n1. **過剰適合**: 学習したモデルは、訓練データ（学習したデータ）の分布に非常に近い形状を維持し、真のデータ（未学習したデータ）の分布と異なる特徴を学習してしまうことがあります。\n2. **精度の低下**: 訓練データだけで得られた精度が、真のデータに対しては大幅に低下します。\n3. **欠陥の増加**: 学習したモデルは、訓練データに含まれる「過剰なパターン」や「ノイズ」に敏感になり、真のデータに適用すると誤りが増加します。\n\n### 過学習の原因\n過学習は、以下の要因が関与します。\n* **欠き欠けのデータ**: 学習対象となるデータが非常に少ない場合（欠き欠けデータ）。\n* **過学習のデータ**: 学習対象,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,LLMがテキストを扱う単位／文字数と常に一致するわけではない,LLM（大規模言語モデル）が文章を扱う際、「トークン」は、**文字列（テキスト）の単位**です。\n\n具体的には、テキストを「文字列」として扱うため、**文字の個数**という単位で定義されます。例えば、日本語の「こんにちは」は 3 つの文字（「こ」、「に」、「ほ」）の 3 つのトークンとして

比較できる質問数: 15


In [ ]:
# @title 用語をそのまま質問：5問
for i, row in enumerate(evaluation_rows[:5]):
    print(f"\n【{i}】{row['質問']}")
    print("正解例:", row["正解例"])
    print("採点ポイント:", row["採点ポイント"])
    print("\n学習前:", row["学習前"])
    print("\n学習後:", row["学習後"])
    print("-" * 60)


【0】トークンとは何ですか？
正解例: LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
採点ポイント: LLMがテキストを扱う単位／単語・文字・その断片などに分割される

学習前: **トークン（Token）** は、インターネット上で通信や処理を行う際に使用される**単一のデータ単位**です。

簡単に言うと、「通信の単位」と「処理の単位」を指します。例えば、Web サイトのユーザー登録時に必要になる「ユーザー名」や、決済システムで使われる「クレジットカード番号」などが、トークンの例です。

トークンがどのように機能するかは、そのシステムによって異なりますが、一般的には以下の 3 つの役割を果たします。

### 1. 認証（認証トークン）
最も一般的な用途は、ユーザーが特定のサービスにログインできるかどうかを確認するものです。
*   **例**: Google の Google アカウント、Facebook の Facebook アカウント、Apple の Apple ID。
*   **仕組み**: 認証トークンは、ユーザーが特定のサービスにログインする際に、そのサービスにアクセスできるかどうかを判断するために使用されます。

### 2. 取引処理（取引トークン）
決済や取引を行う際、取引の完了や確認のために使用されます。
*   **例**: 銀行の ATM での支払い、クレジットカード決済、仮想通貨取引。
*   **仕組み**: 取引が完了した際に発行されるトークン（または取引履歴）は

学習後: LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
------------------------------------------------------------

【1】ハルシネーションとは何ですか？
正解例: LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織

In [ ]:
# @title 言い換えた質問：5問
for i, row in enumerate(evaluation_rows[5:10], start=5):
    print(f"\n【{i}】{row['質問']}")
    print("正解例:", row["正解例"])
    print("採点ポイント:", row["採点ポイント"])
    print("\n学習前:", row["学習前"])
    print("\n学習後:", row["学習後"])
    print("-" * 60)


【5】LLMが文章を扱う際の「トークン」は、どのような単位ですか？
正解例: LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
採点ポイント: LLMがテキストを扱う単位／文字数と常に一致するわけではない

学習前: LLM（大規模言語モデル）が文章を扱う際、「トークン」は、**文字列（テキスト）の単位**です。

具体的には、テキストを「文字列」として扱うため、**文字の個数**という単位で定義されます。例えば、日本語の「こんにちは」は 3 つの文字（「こ」、「に」、「ほ」）の 3 つのトークンとして扱われます。

この「トークン」の単位は、以下の 3 つの要素から構成されます：

1.  **文字（Character）**：テキストに含まれる基本単位。
2.  **言語（Language）**：テキストに含まれる言語の種別。
3.  **順序（Order）**：テキストに含まれる順序関係。

つまり、LLM はテキストを「文字列」として処理し、その文字列の**文字の個数**を「トークン」と呼びます。この「トークン」の個数（例：100 個）は、モデルの出力結果（文脈や文脈の長さ）として扱われます。

学習後: LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
------------------------------------------------------------

【6】AIが事実と異なる内容をもっともらしく答える現象を説明してください。
正解例: LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
採点ポイント: 事実と異なる情報をもっともらしく生成する現象

学習前: AI が事実と異なる内容をもっともらしく答える現象は、**「誤解され

In [ ]:
# @title 用語集にない質問：5問
for i, row in enumerate(evaluation_rows[10:15], start=10):
    print(f"\n【{i}】{row['質問']}")
    print("正解例:", row["正解例"])
    print("採点ポイント:", row["採点ポイント"])
    print("\n学習前:", row["学習前"])
    print("\n学習後:", row["学習後"])
    print("-" * 60)


【10】RAGとは何ですか？
正解例: 質問に関連する情報を外部の文書などから検索し、検索結果を根拠として言語モデルが回答を生成する方法。
採点ポイント: 関連情報を検索する／検索結果を使って言語モデルが回答を生成する

学習前: **RAG（Retrieval-Augmented Generation，検索-Augmented 生成）** は、大規模言語モデル（LLM）の生成能力を「検索（検索結果）」に補完する技術です。

従来の LLM は、文脈や知識を記憶して回答しますが、RAG は**検索エンジン（検索結果）を LLM に提供し、その結果を生成する**仕組みです。

具体的には、以下の 3 つの要素が組み合わさっています。

### 1. 検索エンジン（検索結果）
LLM は、特定の文脈や知識に基づいて回答しますが、その知識が膨大な量に膨らむため、検索結果を**「検索エンジン」**に依存させます。
*   **検索エンジン**は、ユーザーの質問や文脈に基づいて、関連する文章や情報を検索します。
*   **検索結果**は、その検索結果を LLM に提供し、その結果を生成します。

### 2. 検索結果の活用
検索結果を LLM に提供することで、以下のメリットが生まれます。
*   **知識の補完**: 検索結果に含まれる専門用語や知識を LLM に提示することで、

学習後: LLMがLLMではなく、データベースやコードなど別のモデルやツールに渡す入力と、そのモデルが返す出力をまたいで進める thing。安全性と効率を上げるための第一歩。
------------------------------------------------------------

【11】埋め込みベクトルとは何ですか？
正解例: テキストや画像などの特徴を数値のベクトルで表現したもの。ベクトル間の類似度を使い、意味や特徴が近いデータを検索できる。
採点ポイント: データの特徴を数値ベクトルで表す／類似度の計算や検索に利用できる

学習前: **埋め込みベクトル（Embedding Vector）**とは、データ（テキスト、画像、音声など）を「言葉」や「イメージ」に変換して、機械学習モデルに直接入力できる**ベクトル**のことです。

このベクトルは、元のデータの内容を表現

## 6. 用語集の検索結果との比較

モデルによる回答と、用語集を検索して返す回答を比較する。

今回の検索では、質問文に用語集の用語名が含まれているかを調べる。

- 1件見つかった場合：その用語の説明文を返す。
- 見つからない場合：確認が必要と返す。
- 複数見つかった場合：一つに決めず、確認が必要と返す。

この検索は、質問の意味を理解するものではない。
用語名を含まない言い換えでは、説明が用語集にあっても見つからない場合がある。

本評価15問について、学習前・学習後・検索版の回答を比較する。
追加3問は補足実験として分けて確認する。

比較では、正しい回答、誤った回答、確認を求めた回答を区別する。
検索版のneeds_reviewは、検索結果が0件または複数の場合に、
プログラムが付けるフラグである。
モデルが判断したものではなく、Falseでも回答の正しさを保証するものではない。

In [ ]:
# @title 用語集の検索版（本評価15問＋補足3問）
import json
import unicodedata
import pandas as pd

# 全角・半角、大文字・小文字の違いをそろえる
def normalize(text):
    return unicodedata.normalize("NFKC", text).casefold()

def search_glossary(question):
    normalized_question = normalize(question)

    # 質問に用語名が含まれる項目を検索する
    matches = [
        term for term in definitions
        if normalize(term) in normalized_question
    ]

    if not matches:
        return {
            "answer": "該当する用語を検索できませんでした。確認が必要です。",
            "matched_terms": [],
            "needs_review": True,
        }

    # 複数の用語が該当した場合は、自動で一つに決めない
    if len(matches) > 1:
        return {
            "answer": "複数の用語が該当しました。確認が必要です。",
            "matched_terms": matches,
            "needs_review": True,
        }

    term = matches[0]
    return {
        "answer": definitions[term],
        "matched_terms": matches,
        "needs_review": False,
    }

search_rows = []
for ex in test:
    result = search_glossary(ex["question"])
    search_rows.append({
        "category": ex["category"],
        "question": ex["question"],
        "reference_answer": gold[ex["question"]]["reference_answer"],
        "search_answer": result["answer"],
        "matched_terms": result["matched_terms"],
        "needs_review": result["needs_review"],
    })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(search_rows))

search_path = data_path.parent / "glossary_search.json"
search_path.write_text(
    json.dumps(search_rows, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("検索した質問数:", len(search_rows))
print("保存先:", search_path)

,category,question,reference_answer,search_answer,matched_terms,needs_review
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,[トークン],False
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,[ハルシネーション],False
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,[LoRA],False
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,[学習率],False
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,[過学習],False
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,[トークン],False
6,言い換え,AIが事実と異なる内容をもっともらしく答える現象を説明してください。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,該当する用語を検索できませんでした。確認が必要です。,[],True
7,言い換え,LoRAでは、モデル全体を更新する代わりに何を学習するのですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,[LoRA],False
8,言い換え,学習率は、モデルの学習中に何を調整する値ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,[学習率],False
9,言い換え,学習データにはよく答えられるのに、新しいデータで性能が落ちる状態を説明してください。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。,該当する用語を検索できませんでした。確認が必要です。,[],True


検索した質問数: 18
保存先: /content/drive/MyDrive/Colab Notebooks/AIエンジニアリング講座/第2回/発展/glossary_search.json


## 7. 結果の整理と考察

本評価15問について、学習前・学習後・検索版の結果をまとめる。

回答は、正解例と採点ポイントを参考に、次の基準で評価する。

- 2点：必要な内容を説明できており、重大な誤りがない。
- 1点：正しい説明はあるが、必要な内容が不足している。重大な誤りはない。
- 0点：重大な誤りがある、または回答できていない。

確認を求めた回答も内容の得点は0点とするが、
誤った回答とは区別して記録する。
回答が途中で切れている場合も、別に記録する。

採点にはChatGPTの提案を参考にし、最終的な判定は本人が確認する。

### 比較する内容

- 用語を直接尋ねた5問
- 言い換えた5問
- 用語集にない用語を尋ねた5問

種類ごとの得点と、全15問の合計得点を比較する。
追加3問は補足実験として、本評価の得点には含めない。

### 考察する内容

- 学習によって改善した回答と、改善しなかった回答
- 学習後に悪化した回答
- 単純検索で答えられた質問と、答えられなかった質問
- モデルを学習させる方法と、用語集を検索する方法のメリット・デメリット

結論は今回の質問と実験条件の範囲で述べる。

In [1]:
# 保存済みの10エポック版結果を照合・再集計する（再学習なし）
from google.colab import drive
from pathlib import Path
import unicodedata
import json
import pandas as pd
from IPython.display import display

drive.mount("/content/drive")

# フォルダ名の濁点の表現違いにも対応
root = Path("/content/drive/MyDrive/Colab Notebooks")
course_dirs = [
    p for p in root.iterdir()
    if p.is_dir()
    and unicodedata.normalize("NFC", p.name) == "AIエンジニアリング講座"
]
assert len(course_dirs) == 1, "講座フォルダを一意に特定できません"
folder = course_dirs[0] / "第2回" / "発展"

def read_json(name):
    return json.loads((folder / name).read_text(encoding="utf-8-sig"))

saved_scores = read_json("scores.json")
saved_compare = read_json("glossary_compare.json")

assert len(saved_scores) == len(saved_compare) == 15
compare_by_q = {r["質問"]: r for r in saved_compare}
assert len(compare_by_q) == 15

# 質問ごとに学習前後の回答が一致することを確認
for row in saved_scores:
    comparison = compare_by_q[row["質問"]]
    assert row["学習前の回答"] == comparison["学習前の出力"]
    assert row["学習後の回答"] == comparison["学習後の出力"]

restored_df = pd.DataFrame(saved_scores)
columns = ["学習前", "学習後", "検索版"]

restored_summary = restored_df.groupby("区分", sort=False)[columns].sum()
restored_summary.insert(
    0, "満点",
    restored_df.groupby("区分", sort=False).size() * 2
)
restored_summary.loc["全体"] = restored_summary.sum()

# 保存済みCSVとの一致を確認
saved_summary = pd.read_csv(
    folder / "summary.csv", index_col=0, encoding="utf-8-sig"
)
pd.testing.assert_frame_equal(restored_summary, saved_summary)

print("保存済み10エポック版の回答・採点・集計が一致しました。")
print("以下は保存済み結果の再集計です。再学習は行っていません。")
display(restored_summary)

print("学習後に点数が下がった質問")
display(restored_df.loc[
    restored_df["学習後"] < restored_df["学習前"],
    ["質問", "学習前", "学習後"]
])

Mounted at /content/drive
保存済み10エポック版の回答・採点・集計が一致しました。
以下は保存済み結果の再集計です。再学習は行っていません。


,満点,学習前,学習後,検索版
区分,,,,
用語集にある,10,2,10,10
言い換え,10,0,5,6
用語集にない,10,2,0,0
全体,30,4,15,16


学習後に点数が下がった質問


,質問,学習前,学習後
10,RAGとは何ですか？,2,0


In [2]:
# 保存済み10エポック版の回答・採点を使って結果を表示する
# 再学習・回答生成・点数の変更は行わない

scoring_df = restored_df.copy()
summary = restored_summary.copy()

print("保存済み10エポック版の評価結果")

# 質問ごとの回答・採点結果
display(scoring_df)

# 区分ごとの合計点
display(summary)

# 学習後に点数が下がった質問
regressed = scoring_df.loc[
    scoring_df["学習後"] < scoring_df["学習前"],
    ["質問", "学習前", "学習後"],
]
print("学習後に点数が下がった質問")
display(regressed)

保存済み10エポック版の評価結果


,区分,質問,正解例,採点ポイント,学習前の回答,学習前,学習後の回答,学習後,検索の回答,検索版,検索の確認要求
0,用語集にある,トークンとは何ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,LLMがテキストを扱う単位／単語・文字・その断片などに分割される,**トークン（Token）** は、インターネット上で通信や処理を行う際に使用される**単一...,0,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,2,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,2,False
1,用語集にある,ハルシネーションとは何ですか？,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,事実と異なる情報や存在しない情報を生成する／もっともらしい回答になる,**ハルシネーション**（Hulshinēshon）は、物理学、特に**量子力学**において...,0,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,2,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,2,False
2,用語集にある,LoRAとは何ですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,基盤モデルの重みを固定する／更新分を表す低ランクの行列を学習する,**LoRA** は、大規模言語モデル（LLM）のトレーニングにおいて非常に重要な技術の一つ...,0,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,2,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,2,False
3,用語集にある,学習率とは何ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,モデルのパラメータ更新の大きさを調整する,"**学習率（Learning Rate, LR）**とは、機械学習において、モデルがデータか...",0,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,2,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,2,False
4,用語集にある,過学習とは何ですか？,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,学習データへの過度な適合／未知のデータへの性能が低下する,**過学習（Overfitting）**とは、機械学習や統計学習において、モデルが学習したデ...,2,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,2,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,2,False
5,言い換え,LLMが文章を扱う際の「トークン」は、どのような単位ですか？,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,LLMがテキストを扱う単位／文字数と常に一致するわけではない,LLM（大規模言語モデル）が文章を扱う際、「トークン」は、**文字列（テキスト）の単位**で...,0,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,2,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コン...,2,False
6,言い換え,AIが事実と異なる内容をもっともらしく答える現象を説明してください。,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らな...,事実と異なる情報をもっともらしく生成する現象,AI が事実と異なる内容をもっともらしく答える現象は、**「誤解された事実（Falsehoo...,0,LLMが知らないこと（最新情報、組織や講座に固有の情報など）を généralized した...,0,該当する用語を検索できませんでした。確認が必要です。,0,True
7,言い換え,LoRAでは、モデル全体を更新する代わりに何を学習するのですか？,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,基盤モデルの重みを固定する／追加した低ランクの行列を学習する,LoRA（Low-Rank Adaptation）では、モデル全体（全層）を更新するのではな...,0,アダプタの重みを2つの行列の積（低ランク行列）で表し、その重みを更新する学習は行列の積の微分...,0,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、...,2,False
8,言い換え,学習率は、モデルの学習中に何を調整する値ですか？,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,モデルのパラメータ更新の大きさを調整する,学習率は、モデルの学習中に**学習率（learning rate）**を調整する値です。\n...,0,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,2,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定...,2,False
9,言い換え,学習データにはよく答えられるのに、新しいデータで性能が落ちる状態を説明してください。,学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで...,過学習である／学習データへの過度な適合により未知のデータで性能が落ちる,学習データに「よく答えられる」のに、新しいデータで性能が落ちる状態を**「過学習（Overf...,0,過去の学習データに依存し、未知の入力に合う出力を、実際に見られる新しいデータで得られないよう...,1,該当する用語を検索できませんでした。確認が必要です。,0,True


,満点,学習前,学習後,検索版
区分,,,,
用語集にある,10,2,10,10
言い換え,10,0,5,6
用語集にない,10,2,0,0
全体,30,4,15,16


学習後に点数が下がった質問


,質問,学習前,学習後
10,RAGとは何ですか？,2,0
